# Q3 — What is the Impact?
*"Which ecosystems and species are most affected by ocean plastic?"*

## Hypotheses
- **H1:** Marine species in gyre regions show higher plastic ingestion rates than open ocean species
- **H2:** Ghost nets account for a disproportionate share of large marine animal entanglement cases
- **H3:** Different animal groups ingest different types of plastic
- **H4:** Microplastic contamination in commercially consumed fish species represents a measurable risk of human dietary exposure

## Data Sources
- `species.parquet` — marine species observations with ingestion and entanglement records
- `marine_microplastics.parquet` — NOAA NCEI net-tow concentration measurements
- `fish_to_human.parquet` — microplastic counts in commercially consumed fish species

## ML Goal
Can we predict which marine animals are at risk of plastic ingestion based on their species, size, location and local microplastic concentration?

---

## Setup

In [1]:
import sys
sys.path.append("../Src")

import yaml
import numpy as np
import pandas as pd
from q3_marine_life_functions import *

In [2]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

## Data Loading

Load the species observations and microplastics datasets, then clean and prepare both for analysis.
The spatial join (BallTree haversine, 50 km radius) links each species observation to its nearest
microplastic sample point — this enriched dataset (`species_ml`) is used for H1 and the ML model.

In [3]:
species = pd.read_parquet(config["output_data_modular"]["file9"])
microplastics = pd.read_parquet(config["output_data_modular"]["file3"])

In [4]:
species_clean = clean_species(species)
microplastics = clean_microplastics(microplastics)

Species: 10,412 → 10,205 rows after dropping missing coords
Dropped: 207 rows (2.0%)
Microplastics concentration_class after cleaning:
concentration_class
Very Low      5950
Low           2516
Medium       10664
High          2642
Very High      758
Name: count, dtype: int64
Unexpected values: 0


In [5]:
species_ml = spatial_join_species_mp(species_clean, microplastics)

Matched within 50 km : 6,652 / 10,205 (65.2%)
concentration_class
Very Low     2707
Low           588
Medium       2954
High          230
Very High     173
Name: count, dtype: int64


---
## H1 — Marine Species in High-Concentration Areas Show Higher Plastic Ingestion Rates

**Hypothesis:** Animals observed in or near gyre systems — where microplastic concentration is highest —
should show higher rates of plastic ingestion than animals in open ocean areas with lower concentration.

Each species observation is assigned a concentration class (Very Low → Very High) from its matched
microplastic sample. We then compare ingestion rates across classes using Kruskal-Wallis
(non-parametric equivalent of ANOVA — appropriate here given binary/skewed data, multiple groups,
and unequal group sizes) and pairwise Mann-Whitney tests with Bonferroni correction.

In [6]:
ingestion_df = ingestion_rate_by_class(species_ml)
stat, p = run_kruskal_wallis(species_ml)
pairwise_df = run_pairwise_mannwhitney(species_ml)

concentration_class  n_animals  n_with_ingestion  ingestion_rate
           Very Low       2707               199        7.351311
                Low        588                62       10.544218
             Medium       2954               403       13.642519
               High        230                62       26.956522
          Very High        173                12        6.936416
Kruskal-Wallis H = 119.762,  p = 0.0000
→ Significant difference across classes (p < 0.05)
Bonferroni threshold: 0.0050

Very Low vs Low: p_adj=0.0938  —
Very Low vs Medium: p_adj=0.0000  ✓
Very Low vs High: p_adj=0.0000  ✓
Very Low vs Very High: p_adj=1.0000  —
Low vs Medium: p_adj=0.4222  —
Low vs High: p_adj=0.0000  ✓
Low vs Very High: p_adj=1.0000  —
Medium vs High: p_adj=0.0000  ✓
Medium vs Very High: p_adj=0.1152  —
High vs Very High: p_adj=0.0000  ✓


In [7]:
fig_h1 = plot_ingestion_by_class(ingestion_df)
fig_h1.show()

#### H1 Conclusion — ✅ Partially Supported

A spatial join matched 6,652 of 10,205 species observations (65.2%) to a microplastic sample point.
Ingestion rates by concentration class:

| Class | Animals | Ingestion rate |
|---|---|---|
| Very Low | 2,707 | 12.8% |
| Low | 588 | 16.8% |
| Medium | 2,954 | 17.6% |
| High | 230 | **35.6%** |
| Very High | 173 | 13.9% |

Kruskal-Wallis confirmed a significant difference across classes (H = 92.5, p < 0.001).
Pairwise Mann-Whitney tests (Bonferroni corrected) show the **High class is significantly
different from all other classes**.

The relationship is not monotonic — the Very High class drops back to baseline (13.9%),
likely due to small sample size (n = 173) rather than a genuine ecological effect.

> **Limitation:** 34.8% of species observations had no microplastic sample within 50 km
> and were excluded from the analysis.

---
## H2 — Ghost Nets Account for a Disproportionate Share of Large Animal Entanglement

**Hypothesis:** Abandoned fishing nets (ghost nets) preferentially entangle larger marine animals
due to their size, swimming behaviour, and habitat overlap with fishing grounds.

We test this with a chi-square test on the 2×2 contingency table of `is_large` × `ghost_net_entanglement`.
Large animals are defined as `size_avg ≥ 200` (primarily whales, manatees, large turtles, large pinnipeds).

> **Note:** `size_avg` units are inconsistent across groups (likely cm for most species, kg for whales).
> `is_large` should be interpreted as a relative size indicator rather than an absolute measurement.

In [8]:
chi2, p, odds_ratio = run_ghost_net_chi2(species)

Chi-square: 27.319,  p = 0.0000,  df = 1
Odds ratio: 1.423

Entanglement rate — Small: 8.2%,  Large: 11.2%


In [9]:
fig_h2 = plot_ghost_net(chi2, p, odds_ratio)
fig_h2.show()

#### H2 Conclusion — ✅ Confirmed

Chi-square test on the full dataset shows a highly significant association (χ² = 27.3, p < 0.001).

| Size class | Animals | Entanglement rate |
|---|---|---|
| Small (< 200cm) | 4,952 | 8.18% |
| Large (≥ 200cm) | 5,460 | **11.25%** |

Odds ratio = 1.42 — large animals are **42% more likely** to be ghost net entangled than small animals.

The effect is moderate, not absolute — ghost nets affect animals of all sizes. Small animals still
account for the majority of entanglement cases in absolute terms. Size increases risk but is not the only factor.

---
## H3 — Different Animal Groups Ingest Different Types of Plastic

**Hypothesis:** Plastic ingestion profiles reflect each group's habitat and feeding behaviour.
Coastal species should ingest more fishing-related debris; open-ocean seabirds should ingest
more hard plastic fragments mistaken for prey.

Analysis is restricted to the 5 groups with sufficient ingestion records (n ≥ 50):
Manatee, Turtle, Shearwater, Prion, and Toothed Whale.
Rare plastic types (bottle, cloth, net, nurdle) are excluded due to low counts.

In [10]:
profile = build_plastic_profile(species)
fig_h3 = plot_plastic_heatmap(profile)
fig_h3.show()

#### H3 Conclusion — ✅ Confirmed

Plastic type profiles are clearly distinct across groups:

| Group | Dominant plastic type | Likely explanation |
|---|---|---|
| Manatee | Thread, line, fisheries (85–93%) | Coastal habitat, entanglement in fishing debris |
| Turtle | Soft plastic, fisheries (55–70%) | Bags mistaken for jellyfish |
| Toothed Whale | Thread, soft, fisheries (37–56%) | Deep, opportunistic feeding |
| Shearwater | Hard plastic (96%) | Surface feeder mistaking fragments for prey |
| Prion | Hard plastic (96%) | Open ocean seabird — same mechanism as Shearwater |

Seabirds (Prion, Shearwater) are almost exclusively affected by hard plastic fragments,
while marine mammals and turtles show broader profiles dominated by fishing-related debris.

---
## H4 — Microplastic Contamination in Commercial Fish Represents a Risk of Human Dietary Exposure

**Hypothesis:** Commercially consumed fish species carry measurable quantities of microplastics
in their gastrointestinal tracts, creating a direct pathway for human ingestion.

Data compiled from peer-reviewed studies (2020–2023) covers 12 commercially consumed species.
Values represent mean microplastic **particles** found per individual fish.

In [12]:
# ── H4: Load fish-to-human exposure data ─────────────────────────────────
fish_to_human = pd.read_parquet(config["output_data_modular"]["file10"])
fish_sorted = fish_to_human.sort_values("mp_per_individual", ascending=False)
print(fish_sorted[["species", "habitat", "mp_per_individual"]].to_string(index=False))

               species  habitat  mp_per_individual
   Oncorhynchus mykiss   Farmed                9.3
         Sparus aurata Demersal                9.0
  Dicentrarchus labrax Demersal                7.6
       Mullus barbatus Demersal                6.1
  Merlangius merlangus Demersal                4.7
          Gadus morhua Demersal                4.1
   Trachurus trachurus  Pelagic                3.8
       Thunnus thynnus  Pelagic                3.2
    Sardina pilchardus  Pelagic                2.3
Engraulis encrasicolus  Pelagic                1.4


#### H4 Conclusion — ✅ Confirmed

| Finding | Detail |
|---|---|
| Demersal species (bottom dwellers) | Highest MPs — up to 9 particles/individual in Gilthead seabream |
| Farmed species | Up to 9.3 particles/individual in rainbow trout |
| Sardines & anchovies | Lower counts (1.4–2.3) but consumed **whole** — all particles ingested directly |
| Global average | ~5 MPs per individual fish |
| Dominant particle type | Fibers across all species |

An average seafood consumer eating fish several times per week could ingest **thousands of
microplastic particles per year** through fish consumption alone.

---
## Machine Learning Model — Predicting Plastic Ingestion Risk

**Goal:** Can we predict which marine animals are at risk of plastic ingestion based on
species group, body size, location, and local microplastic concentration — using only
information known *before* examining the animal?

**Approach:** Random Forest classifier with `class_weight='balanced'` to handle class imbalance
(17.5% positive rate — ~5 non-ingestion cases per ingestion case).

> **Important — data leakage removed:** The initial model (ROC-AUC 0.989) appeared near-perfect
> because plastic type columns (hard, soft, thread, etc.) were included as features. These record
> plastic *found inside* the animal — essentially encoding the answer directly. All plastic type
> columns were removed. The honest feature set uses only: `group`, `is_large`, `lat`, `lng`,
> `concentration_class`, `gyre_region`.

In [13]:
# ── Encode features and split ───────────────────────────────────────────
df_enc, encoders = encode_features(species_ml)
X_train, X_test, y_train, y_test = split_data(df_enc)

Train: 8,164  |  Test: 2,041
Positive rate — train: 13.1%  |  test: 13.1%


### Train / Test Distribution Check

`stratify=y` ensures the target ratio is preserved, but we also need to verify that
the **feature distributions** are similar between train and test — otherwise the model
could be trained on a biased subset (e.g. most whales in train, mostly turtles in test).

In [14]:
# ── Distribution check: train vs test ───────────────────────────────────
import pandas as pd

print('=== Target balance ===')
print(f'Train positive rate : {y_train.mean():.1%}')
print(f'Test  positive rate : {y_test.mean():.1%}')

print('\n=== Numeric features: mean comparison ===')
num_features = ['is_large', 'latitude', 'longitude']
for col in num_features:
    tr = X_train[col].mean()
    te = X_test[col].mean()
    print(f'{col:20s}  train={tr:.3f}  test={te:.3f}  diff={abs(tr-te):.3f}')

print('\n=== Categorical features: distribution comparison ===')
cat_features = ['group', 'gyre_region', 'concentration_class']
for col in cat_features:
    train_dist = X_train[col].value_counts(normalize=True).sort_index()
    test_dist  = X_test[col].value_counts(normalize=True).sort_index()
    comparison = pd.DataFrame({'train': train_dist, 'test': test_dist}).fillna(0)
    comparison['diff'] = (comparison['train'] - comparison['test']).abs()
    print(f'\n{col}:\n{comparison.round(3)}')

=== Target balance ===
Train positive rate : 13.1%
Test  positive rate : 13.1%

=== Numeric features: mean comparison ===
is_large              train=0.533  test=0.538  diff=0.005
latitude              train=20.454  test=20.807  diff=0.353
longitude             train=-29.456  test=-30.557  diff=1.101

=== Categorical features: distribution comparison ===

group:
       train   test   diff
group                     
0      0.029  0.024  0.005
1      0.002  0.002  0.000
2      0.001  0.001  0.000
3      0.000  0.000  0.000
4      0.000  0.000  0.000
5      0.003  0.004  0.001
6      0.000  0.000  0.000
7      0.003  0.004  0.001
8      0.002  0.003  0.001
9      0.002  0.002  0.000
10     0.484  0.488  0.003
11     0.003  0.004  0.001
12     0.000  0.001  0.001
13     0.005  0.002  0.002
14     0.002  0.001  0.000
15     0.120  0.115  0.005
16     0.036  0.039  0.003
17     0.045  0.047  0.002
18     0.000  0.001  0.001
19     0.004  0.004  0.000
20     0.138  0.128  0.009
21     0.000  

If the `diff` column stays small (< 0.05) across all features, the split is representative
and we can trust that train/test performance differences reflect genuine generalisation,
not a sampling artefact.

In [15]:
# ── Train, tune and compare models ──────────────────────────────────────
rf_base = train_baseline_rf(X_train, y_train)
evaluate_model(rf_base, X_test, y_test, "Baseline Random Forest")

rf_search = tune_rf(X_train, y_train)
rf_tuned = rf_search.best_estimator_
evaluate_model(rf_tuned, X_test, y_test, "Tuned Random Forest")

gb = train_gradient_boosting(X_train, y_train)
evaluate_model(gb, X_test, y_test, "Gradient Boosting")


=== Baseline Random Forest ===
              precision    recall  f1-score   support

No ingestion       0.94      0.89      0.92      1774
   Ingestion       0.48      0.65      0.55       267

    accuracy                           0.86      2041
   macro avg       0.71      0.77      0.73      2041
weighted avg       0.88      0.86      0.87      2041

ROC-AUC : 0.835
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params : {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 18, 'n_estimators': 149}
Best CV AUC : 0.847

=== Tuned Random Forest ===
              precision    recall  f1-score   support

No ingestion       0.96      0.86      0.91      1774
   Ingestion       0.45      0.75      0.56       267

    accuracy                           0.85      2041
   macro avg       0.70      0.80      0.73      2041
weighted avg       0.89      0.85      0.86      2041

ROC-AUC : 0.866

=== Gradient Boosting ===
              precision    recall  f1-scor

0.8561810420176583

#### Feature Importance & Model Evaluation

After removing data leakage, key findings from feature importance:

- **Location dominates** — latitude (40.9%) and longitude (39.1%) together account for 80% of predictive power
- **Species group** contributes 15.2% — consistent with the distinct ingestion profiles found in H3
- **Concentration class, size, gyre region** together only 4.8% — spatial coordinates likely capture the same information more precisely


In [16]:
plot_feature_importance(rf_tuned).show()
plot_roc_curve(rf_tuned, X_test, y_test).show()
plot_probability_distribution(rf_tuned, X_test, y_test).show()

#### Model Summary

| Model | ROC-AUC | Ingestion recall |
|---|---|---|
| Baseline Random Forest | 0.776 | 51% |
| Tuned Random Forest | 0.804 | 59% |
| Gradient Boosting | 0.801 | 62% |
| XGBoost | 0.792 | 64% |

**Selected model: Tuned Random Forest (ROC-AUC 0.804)**

The model is good at *ranking* animals by ingestion risk but not confident enough for definitive
individual predictions — particularly for ingestion cases, which show high uncertainty in the
0.2–0.5 probability range. This reflects genuine biological variability: some animals ingest
plastic regardless of location or species group. The predictive ceiling is set by the available
features, not the model choice.

---
## Q3 Summary

| Hypothesis | Result | Key Finding |
|---|---|---|
| **H1** — High-concentration areas → higher ingestion | ✅ Partial | High class: 35.6% ingestion rate, significantly different from all others (p < 0.001). Non-monotonic — Very High class anomalous (n=173) |
| **H2** — Ghost nets disproportionately entangle large animals | ✅ Confirmed | Odds ratio 1.42 — large animals 42% more likely to be entangled (χ² = 27.3, p < 0.001) |
| **H3** — Animal groups ingest different plastic types | ✅ Confirmed | Seabirds: hard plastic (96%). Marine mammals/turtles: fishing debris (55–93%) |
| **H4** — Commercial fish carry measurable MP risk for humans | ✅ Confirmed | Up to 9.3 MP/individual; whole-fish consumers (sardines, anchovies) ingest all particles |
| **ML** — Predict ingestion risk from observable features | ✅ ROC-AUC 0.804 | Location is the dominant predictor (80%). Individual biological variation limits ceiling |